# Calculate Performance Metrics (Single Model)

## Set Up

### Load packages

In [1]:
# Install packages if not available on your cluster
from sklearn import metrics
import matplotlib.pyplot as plt
from pyspark.sql.functions import col

### Set parameters

In [ ]:
longest_results_model_tb = "catalog.schema.table"
longest_annotated_path = "abfss://container/folder/file.csv"
random_results_model_tb = "catalog.schema.table"
random_annotated_path = "abfss://container/folder/file.csv"

# For additional evaluation subsets, repeat as needed:
# subset_results_model_df = catalog.schema.table
# subset_annotated_df =  "abfss://container/folder/file.csv"

target_col = "MY_TARGET"
id_col_str = "COL1, COL2"
id_col_list = id_col_str.split(",")
id_col_list = [c.strip() for c in id_col_list]

## Load in Model Outputs and Annotated "Ground Truth" Data

In [ ]:
longest_annotated_df = spark.read.csv(
    longest_annotated_path, inferSchema=True, header=True
)
random_annotated_df = spark.read.csv(
    random_annotated_path, inferSchema=True, header=True
)

In [ ]:
longest_results_df = spark.sql(
    "SELECT "
    + id_col_str
    + ", "
    + target_col
    + """, (case when detected_entities <> "" then 1 else 0 end) as predicted_pii_flag FROM """
    + longest_results_model_tb
)


random_results_df = spark.sql(
    "SELECT "
    + id_col_str
    + ", "
    + target_col
    + """, (case when detected_entities <> "" then 1 else 0 end) as predicted_pii_flag FROM """
    + random_results_model_tb
)

## Join and compare model and manual results

In [ ]:
def join_model_and_manual_dfs(model_output_df, manual_annotated_df):

    compare_df = model_output_df.join(
        manual_annotated_df, on=id_col_list, how="inner"
    )
    return compare_df

In [ ]:
compare_longest_df = join_model_and_manual_dfs(
    longest_results_df, longest_annotated_df
)
compare_random_df = join_model_and_manual_dfs(
    random_results_df, random_annotated_df
)

In [ ]:
def compute_and_plot_cm(combined_df):
    actual = combined_df.toPandas()["manual_pii_flag"]
    predicted = combined_df.toPandas()["predicted_pii_flag"]

    cm = metrics.confusion_matrix(actual, predicted, labels=[0, 1])
    cm_display = metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=[0, 1]
    )
    cm_display.plot()
    plt.show()

In [ ]:
compute_and_plot_cm(compare_longest_df)

In [ ]:
compute_and_plot_cm(compare_random_df)

## Inspect PII Missed by Model

In [ ]:
def inspect_missed_pii(combined_df):
    filtered_df = combined_df.where(
        col("manual_pii_flag") > col("predicted_pii_flag")
    )
    filtered_df = filtered_df.select(id_col_list + [target_col])
    return filtered_df

In [ ]:
(inspect_missed_pii(compare_longest_df)).show(truncate=False)

In [ ]:
(inspect_missed_pii(compare_random_df)).show(truncate=False)

## F1, Precision, Accuracy, Recall

Definitions:

- F1 = harmonic mean of precision and recall = 2 * P * R/(P+R)
- Precision = how many of the positive predictions are actually correct = (TP)/(TP + FP)
- Accuracy = proportion of correct predictions out of all predictions = (TP + TN)/(Total Records)
- Recall = how many of the actual positive cases were correctly identified = TP/(TP+FN)

Notes:

For our purpose of PII detection, **recall** is the most important. We want to capture all potential positives for human review; we don't need to be super targeted and only flag true positives.

When there is low prevalence of true PII, **precision** is expected to be low since there are very few positive predictions out there. (e.g. If there is only 1 case of PII and the model flags that 1 case plus 1 other case, then precision is only 0.5.) Precision being low then leads to F1 score being lower too.

In [ ]:
def return_metrics(combined_df):
    actual = (
        combined_df.orderBy(id_col_list).select("manual_pii_flag").collect()
    )
    predicted = (
        combined_df.orderBy(id_col_list).select("predicted_pii_flag").collect()
    )
    my_f1 = metrics.f1_score(actual, predicted)
    my_precision = metrics.precision_score(actual, predicted)
    my_accuracy = metrics.accuracy_score(actual, predicted)
    my_recall = metrics.recall_score(actual, predicted)
    return f"Recall: {my_recall} \nPrecision: {my_precision}  \nAccuracy: {my_accuracy} \nF1: {my_f1} "

In [ ]:
print(return_metrics(compare_longest_df))

In [ ]:
print(return_metrics(compare_random_df))